# 09 — Retrain for FRAMING INVARIANCE (the fix for the unstable score)

**Run on Kaggle with GPU** (Settings → Accelerator → GPU T4 x2 or P100; free 30 hrs/week).

## Why this notebook exists

The shipped detector is **unstable to framing**, and that is now the dominant error source in the
whole product — larger than the thresholds, larger than the scoring formula. Measured with
`scripts/stability_check.py` on one wrecked Civic, same damage, imperceptibly different photos:

| framing | score | classes reported |
|---|---|---|
| original | 38 | crack, missing_part |
| **crop 3%** | **85** | crack |
| crop 6% | 62 | crack |
| **stood back 10%** | **85** | dent, lamp_broken |
| resize 70% | **85** | lamp_broken |

**A 3% crop swings the score 47 points**, and the reported CLASS flips — `missing_part`
(severity 0.28) becomes `lamp_broken` (0.07), a 4× difference. So the score inherits the model's
confusion about *what* the damage is, not merely how confident it is.

**Root cause:** the model was trained on CarDD/VehiDE **close-up damage crops**, but users upload
**whole-car wide shots**. On a wide shot the damage is small in frame, confidence collapses
(measured: the same damage scores 0.33 cropped vs 0.23 wide), and which class wins becomes a
coin flip.

**Four post-processing fixes were already tried and all failed** (multi-scale fusion, softening
the confidence gate, threshold tuning, scoring on coverage). See `docs/CV_FINDINGS.md`. Do not
re-attempt them. Retraining is the only remaining lever.

## What this notebook changes vs notebook 02

1. **Zoom-out augmentation** — synthesises wide shots from close-ups, so the model sees damage
   *small in frame* during training. This targets the exact transform where failure was measured.
2. **Clean-car negatives** — background images with no labels. Measured: the model calls a normal
   wheel `tire_flat` at **0.77** confidence, which scored an undamaged car 38/100. Hard negatives
   are the standard YOLO fix for exactly this.
3. **Stability is the primary metric**, not mAP. mAP is measured on the close-ups this model is
   already good at, so it does not move when the real problem does.
4. **An honest held-out TEST split**, not the validation set the current 0.732 was read from.

## Exit gates — do not ship a model that fails these

| Gate | Current | Required |
|---|---|---|
| Stability range on whole-car photos | **47 pts** | **≤ 15 pts** |
| `tire_flat` false positives on clean wheels | fires at 0.77 | < 0.40 |
| mAP@0.5 on held-out TEST (all 8 classes) | unknown (0.732 is contaminated val, 6/8) | ≥ 0.60 and **published honestly** |
| `crack` recall | 0.389 | ≥ 0.45 |
| Class order in ONNX metadata | — | **byte-identical** to the shipped order |

> **Honest expectation.** Augmentation can only synthesise so much: zooming out of a close-up is
> not the same as a genuine whole-car photograph. Expect this to *reduce* the instability, not
> eliminate it. The complete fix needs real UAE whole-car photos with damage labels — which is
> what the blocked photo pipeline exists to collect. Ship this only if the gates pass.

## 1 · Environment

Pin torch for the GPU Kaggle actually assigns (P100 is sm_60; recent torch drops those kernels).

In [ ]:
# Kaggle may assign a Tesla P100 (sm_60) which recent preinstalled torch no longer builds
# kernels for -> "no kernel image available". Install a torch covering sm_50..sm_90.
!pip -q install "torch==2.4.1" "torchvision==0.19.1" --index-url https://download.pytorch.org/whl/cu121
!pip -q install "ultralytics==8.4.95" "onnx>=1.16" "onnxruntime>=1.18" "onnxslim" opencv-python-headless

import os, random, json, shutil, glob
from pathlib import Path
import numpy as np, torch, cv2

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
import ultralytics; print("ultralytics", ultralytics.__version__)

## 2 · Dataset + the normative class order

**The class order is normative** (`docs/CV_INFERENCE_SPEC.md` §2). It is baked into the ONNX
metadata and `eval/unit_tests.py` asserts the application's list against the artifact. A silent
reorder would remap every detection — a dent priced as a missing part — while every threshold
still looked correct. This cell fails loudly rather than let that happen.

In [ ]:
# NORMATIVE — must not be reordered. Matches the shipped ONNX metadata exactly.
UNIFIED = ["dent", "scratch", "crack", "glass_shatter",
           "lamp_broken", "tire_flat", "punctured", "missing_part"]

candidates = glob.glob("/kaggle/input/**/data.yaml", recursive=True)
print("found data.yaml candidates:")
for c in candidates: print("  ", c)
assert candidates, ("No data.yaml found. Attach the dataset built by notebook 01 "
                    "(CarDD + VehiDE unified) as a Kaggle input first.")

import yaml
SRC_YAML = Path(candidates[0])
src = yaml.safe_load(SRC_YAML.read_text())
src_names = list(src["names"].values()) if isinstance(src["names"], dict) else list(src["names"])
assert src_names == UNIFIED, (
    f"CLASS ORDER MISMATCH — refusing to train.\n  dataset: {src_names}\n  required: {UNIFIED}\n"
    "Fix notebook 01's mapping; do NOT reorder UNIFIED here.")
print("\nclass order verified against the shipped ONNX order")

ROOT = SRC_YAML.parent
print("dataset root:", ROOT)
for split in ("train", "val", "test"):
    imgs = list((ROOT / split / "images").glob("*")) if (ROOT / split / "images").exists() else []
    print(f"  {split}: {len(imgs)} images")

### An honest held-out TEST split

The published 0.732 mAP was read from the **validation** set the model early-stopped on, and
covered only 6 of 8 classes. That number is contaminated and must not be reused. If the dataset
has no `test/` split, carve one deterministically now and never train or early-stop on it.

In [ ]:
WORK = Path("/kaggle/working/ds"); WORK.mkdir(parents=True, exist_ok=True)

def ensure_test_split(root: Path, frac=0.15, seed=SEED):
    # Deterministically carve test/ out of val/ if it does not exist. Never touches train/.
    test_img = root / "test" / "images"
    if test_img.exists() and any(test_img.iterdir()):
        print("test/ already exists — leaving it alone"); return
    val_imgs = sorted((root / "val" / "images").glob("*"))
    assert val_imgs, "no val images to split"
    rng = random.Random(seed); rng.shuffle(val_imgs)
    take = val_imgs[: int(len(val_imgs) * frac)]
    for sub in ("images", "labels"):
        (root / "test" / sub).mkdir(parents=True, exist_ok=True)
    for img in take:
        lbl = root / "val" / "labels" / (img.stem + ".txt")
        shutil.move(str(img), root / "test" / "images" / img.name)
        if lbl.exists(): shutil.move(str(lbl), root / "test" / "labels" / lbl.name)
    print(f"carved {len(take)} images into test/ (deterministic, seed={seed})")

# Copy the dataset into writable working storage before mutating it.
if not (WORK / "data.yaml").exists():
    for split in ("train", "val", "test"):
        s = ROOT / split
        if s.exists(): shutil.copytree(s, WORK / split, dirs_exist_ok=True)
    ensure_test_split(WORK)
print({s: len(list((WORK / s / "images").glob("*"))) for s in ("train", "val", "test")
       if (WORK / s / "images").exists()})

## 3 · Zoom-out augmentation — the core of this retrain

This is what targets the measured failure. The training images are close-ups of damage; the
users' images are whole cars. So synthesise the missing regime: take a training image, **shrink
it into a larger canvas** so the damage occupies a small fraction of the frame, and scale the
YOLO boxes to match.

Because YOLO labels are normalised `cx cy w h`, a zoom-out by factor `z` with the image placed at
offset `(ox, oy)` maps cleanly:

    cx' = ox + cx * z      w' = w * z
    cy' = oy + cy * z      h' = h * z

We keep the originals **and** add zoom-out copies, so the model learns both regimes rather than
trading one for the other.

In [ ]:
ZOOM_FACTORS = [0.55, 0.35]   # damage occupies ~55% / ~35% of the frame — the "stood back" regime
MIN_BOX_FRac = 1e-4           # drop boxes that shrink below usefulness after zoom-out

def zoom_out(img, boxes, z, rng):
    # Shrink img by z into a same-size canvas at a random offset; remap YOLO boxes.
    h, w = img.shape[:2]
    nw, nh = max(1, int(w * z)), max(1, int(h * z))
    small = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    # Fill with a neutral background rather than black: black borders are a strong, unrealistic
    # cue the model can latch onto instead of learning scale invariance.
    canvas = np.full_like(img, 114)
    ox_px, oy_px = rng.randint(0, w - nw), rng.randint(0, h - nh)
    canvas[oy_px:oy_px + nh, ox_px:ox_px + nw] = small
    ox, oy = ox_px / w, oy_px / h
    out = []
    for cls, cx, cy, bw, bh in boxes:
        ncx, ncy, nbw, nbh = ox + cx * z, oy + cy * z, bw * z, bh * z
        if nbw * nbh < MIN_BOX_FRac:      # too small to be learnable — drop, don't teach noise
            continue
        out.append((cls, ncx, ncy, nbw, nbh))
    return canvas, out

def read_labels(p: Path):
    if not p.exists(): return []
    rows = []
    for line in p.read_text().strip().splitlines():
        if not line.strip(): continue
        c, *v = line.split()
        rows.append((int(c), *[float(x) for x in v]))
    return rows

def write_labels(p: Path, boxes):
    p.write_text("".join(f"{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n" for c, cx, cy, w, h in boxes))

rng = random.Random(SEED)
tr_img, tr_lbl = WORK / "train" / "images", WORK / "train" / "labels"
originals = sorted(tr_img.glob("*"))
print(f"{len(originals)} original training images -> adding {len(ZOOM_FACTORS)} zoom-out copies each")

added = skipped = 0
for src_img in originals:
    if "_zoom" in src_img.stem:   # idempotent: never augment an augmentation
        continue
    boxes = read_labels(tr_lbl / (src_img.stem + ".txt"))
    if not boxes:
        continue
    im = cv2.imread(str(src_img))
    if im is None:
        continue
    for z in ZOOM_FACTORS:
        nb_img, nb = zoom_out(im, boxes, z, rng)
        if not nb:
            skipped += 1; continue
        stem = f"{src_img.stem}_zoom{int(z*100)}"
        cv2.imwrite(str(tr_img / f"{stem}.jpg"), nb_img, [cv2.IMWRITE_JPEG_QUALITY, 92])
        write_labels(tr_lbl / f"{stem}.txt", nb)
        added += 1

print(f"added {added} zoom-out images ({skipped} skipped — all boxes shrank below usefulness)")
print("train images now:", len(list(tr_img.glob('*'))))

## 4 · Clean-car negatives — kill the `tire_flat` hallucination

Measured: on a zoomed tile the model calls a **perfectly normal wheel** `tire_flat` at **0.77**,
and that single false positive scored an undamaged car **38/100**. The pipeline currently works
around it by refusing `tire_flat` from tiles (`TILE_EXCLUDE`) — a mask, not a fix.

In YOLO, an image with **no label file** is a background/negative example: the model is penalised
for firing on it. Feeding clean wheels, clean panels and clean whole cars is the direct fix.

> Attach a Kaggle dataset of **clean car photos** (no damage) and point `CLEAN_DIRS` at it.
> Stanford Cars, or any clean-car set, works — the images need no labels at all.

In [ ]:
# TODO: point at clean-car image folders you attached as Kaggle inputs.
CLEAN_DIRS = [p for p in glob.glob("/kaggle/input/**/clean*", recursive=True) if Path(p).is_dir()]
print("clean-car dirs found:", CLEAN_DIRS or "(none — see note below)")

MAX_NEG = 1500   # ~10-15% negatives is the usual sweet spot; too many suppresses recall
neg = 0
for d in CLEAN_DIRS:
    for p in sorted(Path(d).rglob("*")):
        if neg >= MAX_NEG: break
        if p.suffix.lower() not in (".jpg", ".jpeg", ".png"): continue
        im = cv2.imread(str(p))
        if im is None: continue
        cv2.imwrite(str(tr_img / f"neg_{neg:05d}.jpg"), im, [cv2.IMWRITE_JPEG_QUALITY, 92])
        # NO label file written -> YOLO treats this as a background/negative example.
        neg += 1

if neg == 0:
    print("WARNING: no negatives added. The tire_flat false positive will very likely SURVIVE\n"
          "this retrain, and gate 2 below will fail. Attach clean car photos and re-run this cell.")
else:
    print(f"added {neg} clean-car negatives (no labels)")

In [ ]:
DATA_YAML = WORK / "data.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(WORK),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": {i: n for i, n in enumerate(UNIFIED)},   # normative order
}, sort_keys=False))
print(DATA_YAML.read_text())

## 5 · Train

Hyperparameters chosen for **scale invariance**, which is the whole point:

- `scale=0.9` — aggressive random resize (default 0.5). Directly trains the regime that fails.
- `mosaic=1.0` — stitches 4 images, naturally producing small-in-frame objects.
- `close_mosaic=10` — disables mosaic for the last 10 epochs so the model finishes on realistic
  single-image framing rather than on collages.
- `copy_paste=0.3` — more damage instances per image; helps the starved classes (`crack`).
- `fliplr=0.5` — cars are photographed from both sides. `flipud=0.0`: cars are never upside down.
- `deterministic=True` + `seed` — so a re-run reproduces, and a change is attributable.

Keep `imgsz=640`: the shipped ONNX input is **fixed** `[1,3,640,640]` and the browser/backend both
hardcode `IMGSZ = 640`. Changing it is a coordinated change across `cv-browser.ts`, `cv_local.py`
and both version constants — see the note in §8.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")   # COCO-pretrained; same starting point as the shipped model
results = model.train(
    data=str(DATA_YAML),
    epochs=60,
    patience=15,
    imgsz=640,
    batch=-1,              # auto-fit to GPU RAM
    seed=SEED,
    deterministic=True,
    # --- scale/framing invariance: the reason this retrain exists ---
    scale=0.9,
    mosaic=1.0,
    close_mosaic=10,
    copy_paste=0.3,
    translate=0.2,
    fliplr=0.5,
    flipud=0.0,
    degrees=5.0,           # small only — a car photo is rarely rotated much
    project="/kaggle/working/runs",
    name="framing_invariance",
    exist_ok=True,
)
BEST = Path(model.trainer.best)
print("best weights:", BEST)

## 6 · Honest evaluation on the held-out TEST split

Not the validation set. Report **all 8 classes** — the published 0.732 covered 6, which reads as
better than it is because the two omitted classes are the weak ones.

In [ ]:
best = YOLO(str(BEST))
metrics = best.val(data=str(DATA_YAML), split="test", imgsz=640, plots=False)

print(f"\nHELD-OUT TEST  mAP@0.5 {metrics.box.map50:.4f} | mAP@0.5:0.95 {metrics.box.map:.4f}")
print(f"{'class':<16}{'P':>8}{'R':>8}{'mAP50':>9}")
per_class = {}
for i, name in enumerate(UNIFIED):
    try:
        p, r, ap50 = metrics.box.p[i], metrics.box.r[i], metrics.box.ap50[i]
    except (IndexError, TypeError):
        print(f"{name:<16}{'-':>8}{'-':>8}{'-':>9}   (absent from test split)"); continue
    per_class[name] = {"precision": float(p), "recall": float(r), "mAP50": float(ap50)}
    print(f"{name:<16}{p:>8.3f}{r:>8.3f}{ap50:>9.3f}")

classes_reported = len(per_class)
print(f"\nclasses with published P/R: {classes_reported}/8"
      f"{'  <-- report this honestly, do not quote a 6/8 average as overall' if classes_reported < 8 else ''}")

## 7 · Stability — **the metric that decides whether this shipped**

mAP will barely move even if the real problem is fixed, because mAP is measured on the close-ups
the model was always good at. This is the number that matters: take whole-car photos, perturb the
framing imperceptibly, and see whether the detections hold.

This mirrors `scripts/stability_check.py` so the notebook and the repo measure the same thing.

In [ ]:
CONF_GATE = 0.20   # matches cv-browser.ts CONF_THRES / cv_local.CONF_THRES

def variants(im):
    h, w = im.shape[:2]
    yield "original", im
    for p in (3, 6, 10):
        yield f"crop {p}%", im[int(h*p/100):h-int(h*p/100), int(w*p/100):w-int(w*p/100)]
    for p in (10, 25):
        px, py = int(w*p/100), int(h*p/100)
        yield f"back {p}%", cv2.copyMakeBorder(im, py, py, px, px, cv2.BORDER_REPLICATE)
    yield "resize 70%", cv2.resize(im, (int(w*.7), int(h*.7)))

def stability(weights, image_paths, conf=CONF_GATE):
    m = YOLO(str(weights))
    rows = []
    for path in image_paths:
        im = cv2.imread(str(path))
        if im is None: continue
        sets, counts = [], []
        for _, v in variants(im):
            r = m.predict(v, imgsz=640, conf=conf, verbose=False)[0]
            names = {UNIFIED[int(c)] for c in r.boxes.cls.tolist()} if len(r.boxes) else set()
            sets.append(names); counts.append(len(r.boxes))
        union = set().union(*sets) if sets else set()
        inter = set.intersection(*sets) if sets and all(sets) else set()
        # Jaccard over the class sets: 1.0 means every framing agreed on WHICH damage exists.
        agree = len(inter) / len(union) if union else 1.0
        rows.append({"image": Path(path).name, "class_agreement": agree,
                     "box_counts": counts, "union": sorted(union)})
    return rows

# Whole-car photos to test on. Prefer real UAE listing photos; fall back to test-split images.
WHOLE_CAR = sorted(glob.glob("/kaggle/input/**/wholecar*/**/*.jpg", recursive=True))[:20]
if not WHOLE_CAR:
    WHOLE_CAR = [str(p) for p in sorted((WORK / "test" / "images").glob("*"))[:20]]
    print("NOTE: no whole-car set attached — falling back to test-split images, which are\n"
          "close-ups. This UNDERSTATES the instability. Attach real whole-car photos for a\n"
          "meaningful number.")

for label, w in [("OLD (shipped)", "/kaggle/input/shipped-weights/best.pt"), ("NEW", BEST)]:
    if not Path(w).exists():
        print(f"\n{label}: weights not found at {w} — skipping"); continue
    rows = stability(w, WHOLE_CAR)
    if not rows: continue
    mean_agree = sum(r["class_agreement"] for r in rows) / len(rows)
    print(f"\n{label}: mean class agreement across framings = {mean_agree:.3f}  (1.0 = perfect)")
    for r in rows[:5]:
        print(f"   {r['image'][:34]:<34} agree {r['class_agreement']:.2f}  boxes {r['box_counts']}")

## 8 · Export to ONNX — settings are not negotiable

These must match the shipped artifact exactly or the browser breaks:

| setting | value | why |
|---|---|---|
| `imgsz` | 640 | `cv-browser.ts` / `cv_local.py` hardcode `IMGSZ = 640` |
| `dynamic` | `False` | the pipeline assumes a fixed `[1,3,640,640]` input |
| `nms` | `False` | decode + NMS are **ours** (spec §3.4, §3.7); baking them in would double-apply |
| `opset` | 12 | what onnxruntime-web is pinned against |
| `simplify` | `True` | smaller graph, same semantics |

**If you ever change `imgsz` to 1280** you must also: set `IMGSZ` in `cv-browser.ts` *and*
`cv_local.py`, bump `PREPROCESSING_VERSION`, add the new version to
`ACCEPTED_PREPROCESSING_VERSIONS` in `graph/orchestrator.py`, re-run `eval/cv_conformance.py`, and
re-measure peak WASM heap on mobile Safari. It is a coordinated change, not a flag.

In [ ]:
onnx_path = YOLO(str(BEST)).export(
    format="onnx", imgsz=640, opset=12, dynamic=False, nms=False, simplify=True,
)
print("exported:", onnx_path)

import onnxruntime as ort, hashlib
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
meta = sess.get_modelmeta().custom_metadata_map
inp, out = sess.get_inputs()[0], sess.get_outputs()[0]

print("input :", inp.name, inp.shape)
print("output:", out.name, out.shape)

names = eval(meta["names"]) if "names" in meta else {}
exported_order = [names[i] for i in range(len(names))]
print("class order:", exported_order)

assert list(inp.shape) == [1, 3, 640, 640], f"input shape must be [1,3,640,640], got {inp.shape}"
assert list(out.shape) == [1, 12, 8400], f"output shape must be [1,12,8400], got {out.shape}"
assert exported_order == UNIFIED, "CLASS ORDER DRIFTED — do not ship this artifact"
print("\nartifact checks passed")
print("sha256[:12] =", hashlib.sha256(Path(onnx_path).read_bytes()).hexdigest()[:12])

## 9 · Exit gates

Run this last. **If it prints FAIL, do not ship the model** — a retrain that does not move the
stability number has not fixed the reported problem, however good its mAP looks.

In [ ]:
GATES = {
    "mAP@0.5 on held-out TEST >= 0.60": float(metrics.box.map50) >= 0.60,
    "all 8 classes have published P/R": classes_reported == 8,
    "crack recall >= 0.45": per_class.get("crack", {}).get("recall", 0) >= 0.45,
    "ONNX input is [1,3,640,640]": list(inp.shape) == [1, 3, 640, 640],
    "class order unchanged": exported_order == UNIFIED,
}
width = max(len(k) for k in GATES)
for k, ok in GATES.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {k:<{width}}")

print("\nSTABILITY is the deciding gate — compare OLD vs NEW mean class agreement in §7.")
print("Baseline to beat: a 47-point score range / class flips between missing_part and lamp_broken.")
print("\nIf gates pass, ship with:")
print("  1. copy best.onnx to frontend/public/models/ AND cv-service/model/ (they must be identical)")
print("  2. npm run build in frontend/ — scripts/cv-version.mjs regenerates the content-addressed URL")
print("  3. python eval/unit_tests.py      # asserts class order against the NEW artifact")
print("  4. python eval/cv_conformance.py  # browser/backend post-processing parity")
print("  5. python eval/cv_scoring.py      # scoring bands + multi-photo invariants")
print("  6. python scripts/stability_check.py <whole-car photos>   # must be <= 15 points")
print("  7. re-baseline eval/cv_train_summary.json and the /model page — the OLD 0.732 is a")
print("     contaminated val number on 6/8 classes and must not survive this retrain")